In [1]:
import duckdb


In [2]:
# Create a DuckDB connection
con = duckdb.connect(':memory:')

In [8]:
query = """
WITH 
     table_ul_stratifiee_naf AS (
       SELECT siren,
       CASE 
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('01','02','03') THEN 'A'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('05','06','07','08','09') THEN 'B'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('10','11','12','13','14','15','16','17','18',
              '19','20','21','22','23','24','25','26','27','28','29','30','31','32','33') THEN 'C'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('35') THEN 'D'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('36','37','38','39') THEN 'E'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('41','42','43') THEN 'F'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('45','46','47') THEN 'G'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('49','50','51','52','53') THEN 'H'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('55','56') THEN 'I'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('58','59','60','61','62','63') THEN 'J'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('64','65','66') THEN 'K'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('68') THEN 'L'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('69','70','71','72','73','74','75') THEN 'M'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('77','78','79','80','81','82') THEN 'N'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('84') THEN 'O'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('85') THEN 'P'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('86','87','88') THEN 'Q'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('90','91','92','93') THEN 'R'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('94','95','96') THEN 'S'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('97','98') THEN 'T'
              WHEN SUBSTR(activitePrincipaleUniteLegale,1,2) IN ('99') THEN 'U'
              ELSE NULL
       END AS naf_section,
       FROM READ_PARQUET('https://object.files.data.gouv.fr/data-pipeline-open/siren/stock/StockUniteLegale_utf8.parquet')
       GROUP BY siren,naf_section
       ),

     table_ul_stratifiee_nbperiode AS (
       SELECT siren,
              CASE 
                     WHEN count(*)=1 THEN '1'
                     WHEN count(*)<=3 THEN '2-3'
                     WHEN count(*)<=6 THEN '4-6'
                     WHEN count(*)<=10 THEN '7-10' 
                     ELSE '10+'
              END AS nb_periodes, 
       FROM READ_PARQUET('https://object.files.data.gouv.fr/data-pipeline-open/siren/stock/StockUniteLegaleHistorique_utf8.parquet')
       GROUP BY siren
       ),

     table_ul_stratifiee AS (
        SELECT table_ul_stratifiee_nbperiode.siren,
               table_ul_stratifiee_nbperiode.nb_periodes,
               table_ul_stratifiee_naf.naf_section,
        FROM table_ul_stratifiee_nbperiode
        INNER JOIN table_ul_stratifiee_naf ON table_ul_stratifiee_nbperiode.siren=table_ul_stratifiee_naf.siren
        ),

      table_ul_stratifiee_ponderee AS (
        SELECT *,
              ROW_NUMBER() OVER (
                     PARTITION BY nb_periodes,naf_section
                     ORDER BY RANDOM()
              ) AS rn,
              (
              COUNT(*) OVER (
                     PARTITION BY nb_periodes,naf_section
              ) 
              /
              COUNT(*) OVER (
                     PARTITION BY nb_periodes
              ) 
              ) AS naf_poids,
              (
              COUNT(*) OVER () 
              /
              COUNT(*) OVER (
                     PARTITION BY nb_periodes,naf_section
              ) 
              ) AS poids_tirage              
        FROM table_ul_stratifiee
        
       )


SELECT *,
CAST(naf_poids * 
        CASE 
        WHEN nb_periodes = '1' THEN 10000
        WHEN nb_periodes = '2-3' THEN 12500
        WHEN nb_periodes = '4-6' THEN 12500
        WHEN nb_periodes = '7-10' THEN 10000
        ELSE 5000
        END AS INTEGER
    ) AS quota
FROM table_ul_stratifiee_ponderee
WHERE rn <= quota

"""

In [9]:
%%time
test=con.execute(query).fetchdf()


CPU times: total: 1min 4s
Wall time: 18 s


In [10]:
test

,siren,nb_periodes,naf_section,rn,naf_poids,poids_tirage,quota
0,321194540,1,Q,1,0.034172,68.355680,342
1,101775971,1,Q,2,0.034172,68.355680,342
2,522558410,1,Q,3,0.034172,68.355680,342
3,323603191,1,Q,4,0.034172,68.355680,342
4,917695975,1,Q,5,0.034172,68.355680,342
...,...,...,...,...,...,...,...
49997,400621769,10+,F,568,0.114323,3640.288469,572
49998,491122982,10+,F,569,0.114323,3640.288469,572
49999,414207654,10+,F,570,0.114323,3640.288469,572
50000,507908317,10+,F,571,0.114323,3640.288469,572


In [11]:
con.close()